<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs yFinane, which provides a unified interface for loading stock market data and performing financial analysis.

In [ ]:
!pip install yfinance

## Imports and setup

yFinance provides a single entry point for fetching historical stock prices, fundamental data, and economic indicators without juggling multiple API libraries.

In [ ]:
import yfinance as yf

## Load historical price data

We load adjusted closing prices for JPM (our asset) and SPY (our benchmark) over roughly nine years, giving us enough history to observe how the relationship between the two evolves across different market regimes.

In [ ]:
data = yf.download(
    "JPM, SPY",
    start="2014-01-01",
    end="2022-12-31",
)

Choosing SPY as the benchmark matters because the Treynor ratio specifically measures return per unit of market risk. SPY tracks the S&P 500, which is the standard proxy for "the overall market" in U.S. equity analysis. If we picked a sector ETF instead, our beta would measure sensitivity to that sector rather than to broad market movements, and the ratio would answer a different question entirely.

## Compute daily returns and rolling beta

We extract the adjusted close columns into separate series, then convert prices into daily percentage returns. Working with returns instead of raw prices is essential because prices are non-stationary, meaning their statistical properties drift over time and make direct comparisons misleading.

In [ ]:
asset = data.Close.JPM
benchmark = data.Close.SPY
asset_returns = asset.pct_change().dropna()
benchmark_returns = benchmark.pct_change().dropna()

Rolling covariance and variance over a 30-day window let us estimate beta dynamically. A single static beta over nine years would hide the fact that JPM's sensitivity to the market shifts during crises, rate changes, and earnings cycles.

In [ ]:
bm_cov = benchmark_returns.rolling(
    window=30
).cov(asset_returns)

In [ ]:
bm_var = benchmark_returns.rolling(
    window=30
).var()

Beta is the ratio of covariance to variance. This is the standard ordinary least squares slope formula, telling us how many percentage points JPM tends to move for each one-percentage-point move in SPY.

In [ ]:
beta = bm_cov / bm_var

A beta above 1 means JPM amplifies market moves, while a beta below 1 means it dampens them. Because we compute this on a rolling basis, we can watch beta expand during volatile periods and contract during calm ones, which directly affects how we interpret the Treynor ratio over time.

## Calculate and plot the Treynor ratio

We subtract the benchmark return from the asset return to get the excess return, then divide by beta. This is the Treynor ratio: it tells us how much return JPM earns above the market for each unit of systematic risk it carries.

In [ ]:
treynor = (
    asset_returns - benchmark_returns
) / beta

Plotting the ratio as a time series reveals periods where JPM delivered strong risk-adjusted performance and periods where it didn't. Spikes and dips in this chart often align with earnings surprises or macro events, giving us a visual way to separate genuine outperformance from simply riding a bull market. This is the core insight the post builds toward: raw returns can deceive you, but dividing by beta forces you to account for the market risk you were exposed to.

In [ ]:
treynor.plot()

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.